In [1]:
import numpy as np
import pandas as pd
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.metrics import roc_auc_score, accuracy_score, precision_score, f1_score
import matplotlib.pyplot as plt
import warnings
import itertools

# Import oversampling techniques from imblearn
from imblearn.over_sampling import RandomOverSampler, SMOTE, ADASYN
from imblearn.pipeline import Pipeline as ImbPipeline  # Use imblearn's Pipeline

# Import alive-progress for the progress bar
from alive_progress import alive_bar
from contextlib import contextmanager
from joblib.parallel import BatchCompletionCallBack
import threading
from math import prod
from sklearn.exceptions import ConvergenceWarning, UndefinedMetricWarning
import logging

import os
os.environ['PYTHONWARNINGS'] = "ignore"

# Suppress warnings for cleaner output
warnings.filterwarnings("ignore", category=ConvergenceWarning)
warnings.filterwarnings("ignore", category=UndefinedMetricWarning)

# Suppress logging warnings by setting the logging level to ERROR
logging.getLogger().setLevel(logging.ERROR)

# Load the dataset
# Replace the file path with your actual path
data = pd.read_excel("class123_dataset.xlsx")

# Extract the predictors and outcome
X = data.drop('RRI', axis=1)
Y = data['RRI']

# Define the feature indexes to be used as predictors
# Note: Pandas uses 0-based indexing
feature_indexes = [28, 234, 36, 215, 189, 37, 77, 26, 250, 188, 236, 181, 70, 55, 48, 67, 202, 244, 78, 65, 52, 237, 211, 57, 185, 229, 231, 219, 62, 220, 51, 200, 30, 242, 233, 198, 34, 25, 212, 205, 252, 32, 29, 248, 253, 75, 66, 247, 56, 58, 12, 221, 41, 197, 7, 63, 217, 1, 251, 199, 114, 49, 24, 50, 101, 80, 186, 232, 207, 130, 79, 115, 43, 243, 190, 137, 136, 108, 46, 214, 17, 10, 110, 88, 125, 182, 33, 203, 225, 68, 213, 227, 126, 93, 81, 14, 13, 104, 92, 42, 201, 129, 177, 122, 134, 133, 60, 19, 31, 9, 135, 98, 8, 45, 3, 47, 4, 6, 226, 116, 106, 90, 15]

# Select the specified features using .iloc
X_selected = X.iloc[:, feature_indexes]

# Initialize the K-Nearest Neighbors Classifier
knn_classifier = KNeighborsClassifier()

# Define the oversampling strategies
oversamplers = [
    ('random_over_sampler', RandomOverSampler()),
    ('smote', SMOTE()),
    ('adasyn', ADASYN())
]

# Define the pipeline steps
# The pipeline will first apply the oversampler, then the classifier
pipeline_steps = [
    ('oversampler', RandomOverSampler()),  # Placeholder, will be set in GridSearchCV
    ('classifier', knn_classifier)
]

# Create the imbalanced-learn pipeline
pipeline = ImbPipeline(steps=pipeline_steps)

# Define the parameter grid for hyperparameter tuning
# Oversampling grids with conditional logic for 'passthrough'
param_grid_1 = {
    'oversampler': [ADASYN()],
    'oversampler__sampling_strategy': ['auto', 0.1, 0.15, 0.2, 0.25],
    'classifier__n_neighbors': list(range(45, 80)),
    'classifier__weights': ['uniform', 'distance'],
    'classifier__metric': ['euclidean', 'manhattan'],
    'classifier__algorithm': ['auto', 'ball_tree', 'kd_tree', 'brute'],
    'classifier__leaf_size': [20],
}


# Combine the grids
combined_param_grid = [param_grid_1]

# Define the scoring metrics
scoring = {
    'roc_auc': 'roc_auc',
    'accuracy': 'accuracy',
    'precision': 'precision',
    'f1': 'f1'
}

# Define the Stratified 10-Fold Cross-Validation
cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

# Calculate the total number of parameter combinations for the progress bar
total_fits = int(
    sum(
        np.prod([len(values) for values in grid.values()])
        for grid in combined_param_grid
    )
)

@contextmanager
def alive_joblib_bar(total):
    """
    Context manager to integrate alive-progress with joblib's Parallel processing.

    Parameters:
    - total: int, the total number of tasks to be processed.
    """
    with alive_bar(total, title='Grid Search Progress', bar='blocks', force_tty=True) as bar:
        # Store the original BatchCompletionCallBack.__call__ method
        original_callback = BatchCompletionCallBack.__call__
        lock = threading.Lock()

        def on_complete(self, *args, **kwargs):
            with lock:
                bar()
            return original_callback(self, *args, **kwargs)

        # Patch the BatchCompletionCallBack.__call__ method
        BatchCompletionCallBack.__call__ = on_complete
        try:
            yield
        finally:
            # Restore the original method to avoid side effects
            BatchCompletionCallBack.__call__ = original_callback

# Initialize Grid Search with cross-validation
grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid=combined_param_grid,
    scoring=scoring,
    refit='roc_auc',  # Use 'roc_auc' to select the best model
    cv=cv,
    n_jobs=-1,  # Use all available cores
    verbose=0,  # Disable sklearn's verbose
    return_train_score=False
)

# Start the Grid Search with alive-progress
print("Starting Grid Search with Oversampling and Progress Bar...")

with warnings.catch_warnings():
    warnings.filterwarnings("ignore", category=ConvergenceWarning)
    warnings.filterwarnings("ignore", category=UndefinedMetricWarning)
    with alive_joblib_bar(total_fits):
        grid_search.fit(X_selected, Y)

print("Grid Search Completed.")

# Retrieve the best average AUC and its standard deviation
best_auc = grid_search.best_score_
# Retrieve the standard deviation from cv_results_
# Identify the index of the best parameter set
best_index = grid_search.best_index_
best_auc_std = grid_search.cv_results_['std_test_roc_auc'][best_index]

# Retrieve the best hyperparameters
best_params = grid_search.best_params_

# Retrieve the other metrics for the best parameter set
best_accuracy = grid_search.cv_results_['mean_test_accuracy'][best_index]
best_accuracy_std = grid_search.cv_results_['std_test_accuracy'][best_index]

best_precision = grid_search.cv_results_['mean_test_precision'][best_index]
best_precision_std = grid_search.cv_results_['std_test_precision'][best_index]

best_f1 = grid_search.cv_results_['mean_test_f1'][best_index]
best_f1_std = grid_search.cv_results_['std_test_f1'][best_index]

# Output the results
print(f"\nBest Average AUC: {best_auc:.4f} ± {best_auc_std:.4f}")
print(f"Average Accuracy: {best_accuracy:.4f} ± {best_accuracy_std:.4f}")
print(f"Average Precision: {best_precision:.4f} ± {best_precision_std:.4f}")
print(f"Average F1 Score: {best_f1:.4f} ± {best_f1_std:.4f}")
print("\nBest Hyperparameters:")
for param, value in best_params.items():
    if isinstance(value, (SMOTE, ADASYN, RandomOverSampler)):
        print(f"  {param}: {value.__class__.__name__}")
    else:
        print(f"  {param}: {value}")

Starting Grid Search with Oversampling and Progress Bar...
                                                                                [rid Search Progress |▉▉▉▉▉▉▉▉▌                               | ▆█▆ 600/2800 [21Grid Search Progress |▉▉▉▉▉▉▉▉▉▉▉▊                            | ▄▆█ 831/2800 [30Grid Search Progress |▉▉▉▉▉▉▉▉▉▉▉▉▏                           | ▇▇▅ 852/2800 [30Grid Search Progress |▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▏              | ▇▅▃ 1761/2800 [6Grid Search Progress |▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉  | ▅▃▁ 2657/2800 [9Grid Search Progress |▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉✗︎ █▆▄ 3444/2800 [1Grid Search Progress |▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉✗︎ ▆█▆ 3947/2800 [1Grid Search Progress |▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉✗︎ ▃▅▇ 3969/2800 [1Grid Search Progress |▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉✗︎ ▆█▆ 4000/2800 [1Grid Search Progress |▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉✗︎ ▄▂▂ 4010/2800 [1Grid Search Progress |▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉

on 28000: /home/pshw13/myenv/lib/python3.10/site-packages/sklearn/model_selection/_validation.py:540: FitFailedWarning: 
          5600 fits failed out of a total of 28000.
          The score on these train-test partitions for these parameters will be set to nan.
          If these failures are not expected, you can try to debug them by setting error_score='raise'.
          
          Below are more details about the failures:
          --------------------------------------------------------------------------------
          5600 fits failed with the following error:
          Traceback (most recent call last):
            File "/home/pshw13/myenv/lib/python3.10/site-packages/sklearn/model_selection/_validation.py", line 888, in _fit_and_score
              estimator.fit(X_train, y_train, **fit_params)
            File "/home/pshw13/myenv/lib/python3.10/site-packages/sklearn/base.py", line 1473, in wrapper
              return fit_method(estimator, *args, **kwargs)
            File 

Grid Search Progress |▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉✗︎ (!) 28000/2800 [
Grid Search Completed.

Best Average AUC: 0.7525 ± 0.0277
Average Accuracy: 0.8986 ± 0.0082
Average Precision: 0.3952 ± 0.0850
Average F1 Score: 0.2630 ± 0.0525

Best Hyperparameters:
  classifier__algorithm: auto
  classifier__leaf_size: 20
  classifier__metric: manhattan
  classifier__n_neighbors: 51
  classifier__weights: distance
  oversampler: ADASYN
  oversampler__sampling_strategy: 0.2


In [2]:
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_val_predict

# Retrieve the best estimator from Grid Search
best_estimator = grid_search.best_estimator_

# Use cross_val_predict to get cross-validated predicted probabilities
# KNN inherently supports predict_proba
print("\nGenerating cross-validated predicted probabilities...")
y_pred_proba = cross_val_predict(best_estimator, X_selected, Y, cv=cv, method='predict_proba', n_jobs=-1)[:, 1]

# Define a range of threshold values to evaluate
thresholds = np.linspace(0.0, 1.0, 101)

# Initialize variables to store the best metrics and threshold
best_threshold = 0.5
best_f1_score = 0.0
best_accuracy = 0.0
best_precision = 0.0

print("Optimizing threshold to maximize F1 score...")

for threshold in thresholds:
    # Convert predicted probabilities to binary predictions based on the threshold
    y_pred = (y_pred_proba >= threshold).astype(int)
    
    # Calculate F1 score
    current_f1 = f1_score(Y, y_pred)
    
    # Update the best metrics and threshold if current F1 is better
    if current_f1 > best_f1_score:
        best_f1_score = current_f1
        best_threshold = threshold
        best_accuracy = accuracy_score(Y, y_pred)
        best_precision = precision_score(Y, y_pred, zero_division=0)

# Calculate standard deviations using cross-validation
# To compute standard deviations, we'll perform cross-validation predictions and calculate metrics at the best threshold

# Initialize lists to store per-fold metrics
f1_scores = []
accuracies = []
precisions = []

print("\nCalculating metrics at the optimal threshold across folds...")

for fold, (train_idx, test_idx) in enumerate(cv.split(X_selected, Y), 1):
    # Split data
    X_train, X_test = X_selected.iloc[train_idx], X_selected.iloc[test_idx]
    y_train, y_test = Y.iloc[train_idx], Y.iloc[test_idx]
    
    # Fit the model on the training data
    best_estimator.fit(X_train, y_train)
    
    # Predict probabilities on the test data
    y_proba_fold = best_estimator.predict_proba(X_test)[:, 1]
    
    # Apply the optimal threshold
    y_pred_fold = (y_proba_fold >= best_threshold).astype(int)
    
    # Calculate metrics
    fold_f1 = f1_score(y_test, y_pred_fold)
    fold_accuracy = accuracy_score(y_test, y_pred_fold)
    fold_precision = precision_score(y_test, y_pred_fold, zero_division=0)
    
    # Append to lists
    f1_scores.append(fold_f1)
    accuracies.append(fold_accuracy)
    precisions.append(fold_precision)
    
    print(f"  Fold {fold}: F1={fold_f1:.4f}, Accuracy={fold_accuracy:.4f}, Precision={fold_precision:.4f}")

# Calculate mean and standard deviation for the metrics
mean_f1 = np.mean(f1_scores)
std_f1 = np.std(f1_scores)

mean_accuracy = np.mean(accuracies)
std_accuracy = np.std(accuracies)

mean_precision = np.mean(precisions)
std_precision = np.std(precisions)

# Output the optimized threshold and corresponding metrics
print(f"\n=== Optimized Threshold ===")
print(f"Threshold for Maximum F1 Score: {best_threshold:.2f}")

print(f"\n=== Metrics at Optimal Threshold ===")
print(f"F1 Score: {mean_f1:.4f} ± {std_f1:.4f}")
print(f"Accuracy: {mean_accuracy:.4f} ± {std_accuracy:.4f}")
print(f"Precision: {mean_precision:.4f} ± {std_precision:.4f}")


Generating cross-validated predicted probabilities...
Optimizing threshold to maximize F1 score...

Calculating metrics at the optimal threshold across folds...
  Fold 1: F1=0.3942, Accuracy=0.8659, Precision=0.3375
  Fold 2: F1=0.3360, Accuracy=0.8657, Precision=0.3043
  Fold 3: F1=0.2791, Accuracy=0.8495, Precision=0.2466
  Fold 4: F1=0.2946, Accuracy=0.8528, Precision=0.2603
  Fold 5: F1=0.2920, Accuracy=0.8430, Precision=0.2469
  Fold 6: F1=0.2535, Accuracy=0.8285, Precision=0.2093
  Fold 7: F1=0.3968, Accuracy=0.8770, Precision=0.3571
  Fold 8: F1=0.3770, Accuracy=0.8770, Precision=0.3538
  Fold 9: F1=0.2857, Accuracy=0.8544, Precision=0.2609
  Fold 10: F1=0.3582, Accuracy=0.8608, Precision=0.3117

=== Optimized Threshold ===
Threshold for Maximum F1 Score: 0.32

=== Metrics at Optimal Threshold ===
F1 Score: 0.3267 ± 0.0496
Accuracy: 0.8575 ± 0.0143
Precision: 0.2888 ± 0.0485
